# Ch 21. RLHF — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: REINFORCE로 CartPole 같은 toy 환경을 학습하라.

### 해설

외부 환경 의존성을 피하기 위해 두 행동 bandit을 toy 정책으로 사용한다. REINFORCE 추정량은 $-(R-b)\log\pi(a)$를 최소화하며, 베이스라인은 기대값을 바꾸지 않고 분산을 줄인다. 아래 고정 시드 실행은 더 좋은 행동의 확률이 증가함을 확인한다.


In [ ]:
import numpy as np

rng=np.random.default_rng(2101); logits=np.zeros(2); rewards=np.array([0.2,0.8]); history=[]
for step in range(800):
    probs=np.exp(logits-logits.max()); probs/=probs.sum(); action=rng.choice(2,p=probs)
    reward=rewards[action]+rng.normal(scale=.05); advantage=reward-probs@rewards
    grad=-probs; grad[action]+=1; logits += .08*advantage*grad
    if step%100==0: history.append(float(probs[1]))
final=np.exp(logits-logits.max()); final/=final.sum()
assert final[1] > .9 and history[-1] > history[0]
print({"probability_better_action":round(float(final[1]),4),"learning_curve":np.round(history,3).tolist()})


## 문제 2

**문제**: PPO에서 $\epsilon = 0.1, 0.2, 0.3$을 비교하고 영향을 분석하라.

### 해설

$r=\pi_\theta/\pi_{old}$에서 clip 구간은 $[1-\epsilon,1+\epsilon]$이다. 작은 $\epsilon$은 업데이트를 보수적으로 만들고 큰 값은 더 큰 변화와 불안정 가능성을 허용한다. 아래는 같은 $r,A$에서 목적값이 어떻게 제한되는지 정확히 비교한다.


In [ ]:
import numpy as np

ratios=np.linspace(.6,1.4,401); advantages=np.ones_like(ratios); results={}
for epsilon in (.1,.2,.3):
    surrogate=np.minimum(ratios*advantages,np.clip(ratios,1-epsilon,1+epsilon)*advantages)
    clipped_fraction=float(np.mean(ratios>1+epsilon)); max_objective=float(surrogate.max())
    results[epsilon]={"clipped_fraction":clipped_fraction,"max_surrogate":max_objective}
assert results[.1]["clipped_fraction"] > results[.2]["clipped_fraction"] > results[.3]["clipped_fraction"]
print({str(k):{m:round(v,4) for m,v in r.items()} for k,r in results.items()})


## 문제 3

**문제**: RM 학습에서 chosen과 rejected가 비슷할 때 vs 다를 때 loss를 비교하라.

### 해설

RM loss는 $-\log\sigma(r_c-r_r)$이다. 점수 차가 0이면 $\log2$, chosen이 크게 우세하면 0에 접근하고 순서가 틀리면 커진다. 이는 절대 점수가 아니라 차이만 식별됨을 뜻한다.


In [ ]:
import numpy as np

gaps=np.array([0.0,0.5,2.0,-0.5]); losses=np.logaddexp(0,-gaps)
assert losses[2] < losses[1] < losses[0] < losses[3]
print([{"chosen_minus_rejected":float(g),"preference_loss":round(float(l),5)} for g,l in zip(gaps,losses)])


## 문제 4

**문제**: KL 패널티 $\beta = 0, 0.1, 1.0$에서 정책 변화를 시뮬레이션하라.

### 해설

$R_{total}=R_{RM}-\beta D_{KL}$에서 $\beta$가 커질수록 참조 정책에서 멀어지는 이득이 억제된다. 아래의 1차원 오목 목적은 최적 변화량이 $1/\beta$에 비례함을 보이며 $\beta=0$은 유한한 제약 최적점이 없다.


In [ ]:
import numpy as np

# Optimize reward*delta - beta*delta^2/2 on a bounded grid; delta proxies policy movement/KL.
delta=np.linspace(0,3,3001); results={}
for beta in (0.0,.1,1.0):
    objective=delta-beta*delta**2/2
    optimum=float(delta[np.argmax(objective)])
    results[beta]={"optimal_policy_shift":optimum,"kl_proxy":optimum**2/2}
assert results[1.0]["optimal_policy_shift"] < results[.1]["optimal_policy_shift"] <= results[0.0]["optimal_policy_shift"]
print({str(k):{m:round(v,3) for m,v in r.items()} for k,r in results.items()})


## 문제 5

**문제**: RLHF 대비 DPO의 장점을 메모리 관점에서 설명하라.

### 해설

PPO식 RLHF 학습은 정책, 참조, 보상, 가치 모델과 rollout 활성값을 요구한다. DPO는 보통 학습 정책과 고정 참조의 로그확률만 필요하여 보상·가치 모델과 온라인 rollout을 제거한다. 따라서 모델 상태와 활성값 메모리가 감소한다.


In [ ]:
# Count simultaneously resident trainable/frozen model-sized states under explicit assumptions.
ppo={"policy":1,"reference":1,"reward":1,"value":1,"rollout_buffer":.25}
dpo={"policy":1,"reference":1,"preference_batch":.05}
ppo_total=sum(ppo.values()); dpo_total=sum(dpo.values())
assert dpo_total < ppo_total and "reward" not in dpo and "value" not in dpo
print({"units_of_model_memory":{"PPO":ppo_total,"DPO":dpo_total},"relative_reduction":round(1-dpo_total/ppo_total,3),
       "assumption":"same precision; optimizer/activation memory excluded"})
